In [ ]:
from pathlib import Path
import json, re
from owlready2 import *

JSON_PATH = Path("ontology/swrl_rules.json")
ONTO_PATH = Path("ontology/core.ttl")

In [ ]:
def slug(s: str) -> str:
	return re.sub(r'[^a-z0-9_]+', '_', s.lower()).strip('_')

def load_rules_from_json(onto: Ontology, json_path: Path) -> list[Imp]:
	data = json.loads(json_path.read_text(encoding="utf-8"))
	if "prefixes" in data:
		for pfx, iri in data["prefixes"].items():
			onto.world.get_namespace(iri)

	created: list[Imp] = []
	with onto:
		for item in data.get("rules", []):
			rule_id   = slug(item.get("id") or item.get("label") or "rule")
			rule_text = item["rule"]
			label     = item.get("label")
			
			imp = Imp(rule_id)
			if label:
				imp.label = [label]

			try:
				imp.set_as_rule(rule_text)  # parse della sintassi SWRL compatta
				created.append(imp)
				print(f"[OK] {rule_id}")
			except Exception as e:
				# In caso di errore di parsing, rimuoviamo l’Imp per non sporcare l’ontologia
				try: imp.destroy()
				except: pass
				print(f"[ERR] {rule_id}: {e}")

	return created

In [ ]:
world = World()
onto  = world.get_ontology(ONTO_PATH).load()
imps = load_rules_from_json(onto, JSON_PATH)
print("Regole caricate: {len(imps)}")

with onto:
	sync_reasoner_pellet(infer_property_values=True, infer_data_property_values=True)

onto.save(file="out.ttl", format="turtle")
print(f"Salvato out.ttl")
